# 08 · Does L_mul really order events better? · Tuned penalties

The first results showed L_mul models scoring *above the oracle* on `within_ctd` in the
semi-competing settings, so that metric can be gamed. Here:

* **Stage A** scores within-subject ordering against the simulator's **true (latent)
  event times** with proper scores (`true_order_*`, `scripts/true_order.py`), on the base
  (non-competing) and the semi-competing scenario — Oracle, Cox, DeepHit, MENSA and our
  model with L_PCH / +L_mul / +L_rank+L_mul, 5 splits.
* **Stage B** tunes the penalty weight and σ of L_rank (METABRIC) and L_mul (sim_base)
  on validation C_td, then compares the tuned penalty with L_PCH on 5 splits — so a
  negative result cannot be blamed on untuned weights.

## Options

In [ ]:
NAME = "08_true_order_and_tuning"

# Stage A - within-subject ordering against the TRUE event order (scripts/true_order.py;
# computed automatically for every run on simulated data).
TRUTH_DATASETS = ["sim_base", "sim_regime_semicompeting"]
TRUTH_BASELINES = ["coxph", "deephit_paper", "mensa_paper"]
TRUTH_RECIPES = ["nllpch", "nllpch_event_ranking", "nllpch_sample_event_ranking"]

# Stage B - give the penalties a fair chance: tune weight and sigma on validation C_td
# (split 0, seed 0), then test the chosen setting on 5 splits against L_PCH.
TUNE = {
    # penalty recipe, dataset, {config key: values}
    "L_rank": ("nllpch_sample_ranking", "metabric",
               {"ranking_loss_sigma": [0.1, 0.5, 1.0], "ranking_loss_coeff": [0.01, 0.05, 0.2]}),
    "L_mul":  ("nllpch_event_ranking", "sim_base",
               {"event_ranking_loss_sigma": [0.1, 0.5, 1.0], "event_ranking_loss_coeff": [0.01, 0.05, 0.2]}),
}
SEEDS = [0, 1, 2, 3, 4]

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/version_1/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Datasets

In [ ]:
# ---- datasets -------------------------------------------------------------------------
def ensure_dataset(ds):
    """Simulated scenarios and EBMT are generated on the fly; hsa_synthetic ships with the repo."""
    if ds == "hsa_synthetic" or (REPO / "data" / ds / "meta.json").is_file():
        return
    if ds == "ebmt":
        subprocess.run([sys.executable, "scripts/prepare_ebmt.py"], check=True)
    elif ds.startswith("sim_"):
        subprocess.run([sys.executable, "-m", "sat.data.simulate", "--scenario", ds[4:],
                        "--out", f"data/{ds}"], check=True, stdout=subprocess.DEVNULL)
    else:
        raise ValueError(ds)

def group(ds):
    return "hsa_synthetic" if ds == "hsa_synthetic" else "multievent"

def ds_overrides(ds):
    # hsa keeps its own configs; score every event at its own horizons like the others
    return ["per_event_horizons=true"] if ds == "hsa_synthetic" else dataset_overrides(ds, REPO)

def has_truth(ds):
    return (REPO / "data" / ds / "truth.npz").is_file()

SINGLE = {"metabric": ("survtrace_metabric", "metabric_numeric")}
def g8(ds):  return SINGLE[ds][0] if ds in SINGLE else group(ds)
def hub8(ds): return SINGLE[ds][1] if ds in SINGLE else ds
def ov8(ds): return [] if ds in SINGLE else ds_overrides(ds)

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    TRUTH_DATASETS = TRUTH_DATASETS[:1]
    TUNE = {k: (r_, d_, {kk: vv[:1] for kk, vv in grid.items()}) for k, (r_, d_, grid) in TUNE.items()}
for ds in sorted(set(TRUTH_DATASETS) | {d for _, d, _ in TUNE.values()}):
    if ds not in SINGLE:
        ensure_dataset(ds)
    r.prepare(f"{g8(ds)}/survival", ov8(ds), label=ds)

## Stage A · ordering against the true event order

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for ds in TRUTH_DATASETS:
        g, dov, base = group(ds), ds_overrides(ds), {"dataset": ds, "seed": s, "stage": "truth"}
        runs.append(Run(f"{ds}__oracle__s{s}", "oracle", f"{g}/survival", dov + so, ds, base | {"model": "oracle"}))
        for m in TRUTH_BASELINES:
            runs.append(Run(f"{ds}__{m}__s{s}", "coxph" if m == "coxph" else "finetune",
                            f"{g}/{m}", dov + so, ds, base | {"model": m}))
        for rec in TRUTH_RECIPES:
            runs.append(Run(f"{ds}__ours_{rec}__s{s}", "finetune", f"{g}/survival",
                            dov + so + [f"tasks/losses={rec}"], ds, base | {"model": "ours", "recipe": rec}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Stage B · tune the penalties (validation only), then test on 5 splits

In [ ]:
import itertools
BEST = {}
for pen, (rec, ds, grid) in TUNE.items():
    keys = list(grid)
    tune_runs = []
    for i, vals in enumerate(itertools.product(*grid.values())):
        ov = [f"{k}={v}" for k, v in zip(keys, vals)] + ["seed=0", "split_seed=0", f"tasks/losses={rec}"]
        tune_runs.append(Run(f"tune_{pen}__{ds}__{i:02d}", "finetune", f"{g8(ds)}/survival", ov8(ds) + ov,
                             hub8(ds), {"dataset": ds, "model": "tune", "penalty": pen, **dict(zip(keys, vals))}))
    r.run_many(tune_runs, time_budget_min=TIME_BUDGET_MIN)
    val = r.collect(split="validation")
    val = val[(val.get("model") == "tune") & (val.get("penalty") == pen)]
    if len(val):
        best = val.sort_values("ctd_weighted_avg", ascending=False).iloc[0]
        BEST[pen] = [f"{k}={best[k]:g}" for k in keys]
        display(val[keys + ["ctd_weighted_avg"]].sort_values("ctd_weighted_avg", ascending=False)
                .rename(columns={"ctd_weighted_avg": "val C_td"}))
(r.results_dir / "tuned_penalties.json").write_text(json.dumps(BEST, indent=2))
print("chosen:", BEST)

runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for pen, (rec, ds, _) in TUNE.items():
        base = {"dataset": ds, "seed": s, "stage": "tuned", "model": "ours"}
        runs.append(Run(f"tuned__{ds}__ours_nllpch__s{s}", "finetune", f"{g8(ds)}/survival",
                        ov8(ds) + so + ["tasks/losses=nllpch"], hub8(ds), base | {"recipe": "nllpch"}))
        runs.append(Run(f"tuned__{ds}__ours_{rec}__s{s}", "finetune", f"{g8(ds)}/survival",
                        ov8(ds) + so + [f"tasks/losses={rec}"] + BEST.get(pen, []), hub8(ds),
                        base | {"recipe": rec, "tuned": " ".join(BEST.get(pen, []))}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
* Stage A: **True-order C / Brier / log-loss** (proper scores against the latent event times; the oracle is the ceiling and cannot be beaten in expectation) next to the gameable `within_ctd`. If `L_mul` really orders events better it must win here.
* Stage B: `ablation_paired` = tuned penalty vs L_PCH on the same 5 splits.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)